# 話者分離つき時刻付け align版 (Colab Pro GPU用)

対象: Googleレコーダー由来m4a + Pixel版テキスト正 (30分-2時間、中日混じり)。
方針: 全文デコードを省略し、stable-ts強制アライメントで時刻付けのみ行う。デコード誤差と時間を削る高速化PoC。
構成: stable-ts align (turbo) + sherpa-onnx分離 (HFトークン不要)。
入力: Pixel版txtは必須。zip同梱かto_colab内txtを自動検出する。なければ停止する。
出力: align_ 接頭辞つきで保存し、v2成果物と衝突しない。


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("GPUなしのため停止。ランタイム→タイプ変更でGPUにすること")
print(torch.cuda.get_device_name(0))
!pip install -q stable-ts sherpa-onnx soundfile numpy


In [ ]:
from google.colab import files
from pathlib import Path

# 伝書鳩回避: Drive直結。TrueならDriveの入出力を使い、上げ下げ不要
USE_DRIVE = True
DRIVE_BASE = "/content/drive/MyDrive/フロニシス/Gdrive_code"  # Kaiの窓。変える時はここだけ
TO_COLAB_DIR = DRIVE_BASE + "/to_colab"  # 入力置き場
DRIVE_OUTDIR = DRIVE_BASE + "/from_colab"  # 出力先。同期フォルダに合わせる
DRIVE_IN = ""  # 空ならto_colab内の音声を自動検出。複数あれば先頭1件
PIXEL_TXT = ""  # 空なら自動検出。zip同梱またはto_colab内txt。なければ停止
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTBASE = DRIVE_OUTDIR
    if not DRIVE_IN:
        if not Path(TO_COLAB_DIR).is_dir():
            raise SystemExit(f"入力置き場なし: {TO_COLAB_DIR}。Driveマウントとフォルダ名を確認すること")
        cands = sorted([str(p) for p in Path(TO_COLAB_DIR).glob("*") if p.suffix.lower() in (".m4a", ".mp3", ".wav", ".zip")])
        if not cands:
            raise SystemExit("to_colabに音声なし。m4a/mp3/wav/zipを置くこと")
        if len(cands) > 1:
            print("複数検出、先頭を使用:", cands)
        DRIVE_IN = cands[0]
    if DRIVE_IN.lower().endswith(".zip"):
        import zipfile
        with zipfile.ZipFile(DRIVE_IN) as z:
            z.extractall("/content/unzipped")
        auds = sorted([str(p) for p in Path("/content/unzipped").rglob("*") if p.suffix.lower() in (".m4a", ".mp3", ".wav")])
        txts = sorted([str(p) for p in Path("/content/unzipped").rglob("*.txt")])
        if not auds:
            raise SystemExit("zip内に音声なし")
        if not txts:
            raise SystemExit("zip内にPixel版txtなし。音声とtxtを同梱したzipを置くこと")
        print("zip解凍。Pixel版:", txts[0])
        if not PIXEL_TXT:
            PIXEL_TXT = txts[0]
        DRIVE_IN = auds[0]
    else:
        if not PIXEL_TXT:
            same = str(Path(TO_COLAB_DIR) / (Path(DRIVE_IN).stem + ".txt"))
            if Path(same).is_file():
                PIXEL_TXT = same
            else:
                txts = sorted([str(p) for p in Path(TO_COLAB_DIR).glob("*.txt")])
                if not txts:
                    raise SystemExit("Pixel版txtなし。to_colabに音声と同名txtかtxtを1件置くこと")
                if len(txts) > 1:
                    print("txt複数検出、先頭を使用:", txts)
                PIXEL_TXT = txts[0]
    SRC = DRIVE_IN
else:
    up = files.upload()
    names = list(up.keys())
    auds = [n for n in names if Path(n).suffix.lower() in (".m4a", ".mp3", ".wav")]
    txts = [n for n in names if n.lower().endswith(".txt")]
    if not auds:
        raise SystemExit("音声なし。m4a/mp3/wavを上げること")
    if not txts and not PIXEL_TXT:
        raise SystemExit("Pixel版txtなし。音声とtxtを両方上げること")
    SRC = auds[0]
    if not PIXEL_TXT:
        PIXEL_TXT = txts[0]
    OUTBASE = "/content/out"
print("入力:", SRC)
print("Pixel版:", PIXEL_TXT)

# 設定 (必要なら変える)
LANGUAGE = "ja"  # auto/ja/zh。align時の言語指定に使う
NUM_SPEAKERS = 0  # 人数が分かれば固定 (37分裂の特効薬)。不明なら0
CLUSTER_THRESHOLD = 0.5  # 小さいほど話者数が増える


In [ ]:
!ffmpeg -y -v error -i "$SRC" -ac 1 -ar 16000 /content/in_16k.wav
!ls -lh /content/in_16k.wav
# 分離モデル取得 (一次情報: k2-fsa/sherpa-onnx releases。綴りrecongitionは公式通り)
!wget -q https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-segmentation-models/sherpa-onnx-pyannote-segmentation-3-0.tar.bz2
!tar xf sherpa-onnx-pyannote-segmentation-3-0.tar.bz2
!wget -q https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/nemo_en_titanet_small.onnx
!ls -lh sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx nemo_en_titanet_small.onnx


In [ ]:
import soundfile as sf
import numpy as np
import json
from pathlib import Path
import stable_whisper

data, sr = sf.read("/content/in_16k.wav", dtype="float32", always_2d=True)
assert sr == 16000
audio = np.ascontiguousarray(data.mean(axis=1), dtype=np.float32)
total_s = len(audio) / 16000
print(f"音声 {total_s/60:.1f}分")

pixel_txt = Path(PIXEL_TXT).read_text(encoding="utf-8").strip()
if not pixel_txt:
    raise SystemExit(f"Pixel版txtが空: {PIXEL_TXT}")
print(f"Pixel版 {len(pixel_txt)}文字")

# 言語解決: jaはJapanese系に寄せる。autoは検出に任せる
langmap = {"ja": "Japanese", "zh": "Chinese", "en": "English"}
if LANGUAGE == "auto":
    align_lang = "auto"
elif LANGUAGE in langmap:
    align_lang = langmap[LANGUAGE]
else:
    align_lang = LANGUAGE

try:
    model = stable_whisper.load_model("turbo")
    if align_lang == "auto":
        result = model.align("/content/in_16k.wav", pixel_txt, language=None)
    else:
        result = model.align("/content/in_16k.wav", pixel_txt, language=align_lang)
except Exception as e:
    print(f"アライメント失敗: {type(e).__name__}: {e}")
    print("対処: Pixel版txtの文字化け・空行のみ・音声不一致を確認すること")
    raise SystemExit(f"アライメント失敗のため停止: {type(e).__name__}: {e}")

# v2互換wsegs (start/end/text/確信度0)。握りつぶしなし
wsegs = []
for s in result.segments:
    is_dict = isinstance(s, dict)
    text = str(s.get("text", "") if is_dict else getattr(s, "text", "")).strip()
    start = float(s.get("start", 0.0) if is_dict else getattr(s, "start", 0.0))
    end = float(s.get("end", 0.0) if is_dict else getattr(s, "end", 0.0))
    if not text:
        continue
    if end < start:
        end = start
    wsegs.append({"start": start, "end": end, "text": text,
                  "avg_logprob": 0.0, "no_speech_prob": 0.0})
if not wsegs:
    raise SystemExit("アライメント結果が空。Pixel版txtと音声の対応を確認すること")
print(f"アライメント完了: {len(wsegs)}件 (確信度はalign経路のため0固定)")


In [ ]:
import time
from pathlib import Path
import sherpa_onnx

# 分離は一括で行う (チャンク毎だと話者番号が割れるため)。2時間でも数百MBで収まる
seg_model = "sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx"
if not Path(seg_model).exists():
    seg_model = "sherpa-onnx-pyannote-segmentation-3-0/model.onnx"
seg_cfg = sherpa_onnx.OfflineSpeakerSegmentationModelConfig(
    pyannote=sherpa_onnx.OfflineSpeakerSegmentationPyannoteModelConfig(model=seg_model))
emb_cfg = sherpa_onnx.SpeakerEmbeddingExtractorConfig(model="nemo_en_titanet_small.onnx")
clu_cfg = sherpa_onnx.FastClusteringConfig(num_clusters=NUM_SPEAKERS, threshold=CLUSTER_THRESHOLD)
cfg = sherpa_onnx.OfflineSpeakerDiarizationConfig(
    segmentation=seg_cfg, embedding=emb_cfg, clustering=clu_cfg,
    min_duration_on=0.3, min_duration_off=0.5)
dz = sherpa_onnx.OfflineSpeakerDiarization(cfg)
print(f"分離開始 ({Path(seg_model).name})", flush=True)
t0 = time.time()
last = [-1]
def cb(done, total):
    pct = int(done * 100 / max(1, total))
    if pct >= last[0] + 5:
        last[0] = pct - (pct % 5)
        print(f"分離 {pct}% ({done}/{total})", flush=True)
    return 0
dsegs_raw = dz.process(audio, callback=cb).sort_by_start_time()
print(f"分離完了 {time.time()-t0:.0f}秒", flush=True)
dsegs = [{"start": float(s.start), "end": float(s.end), "speaker": f"SPEAKER_{s.speaker:02d}", "confidence": float(getattr(s, "confidence", 0.0))} for s in dsegs_raw]
print(f"分離: {len(dsegs)}区間")

# 時刻重なりで結合 + 幻覚除去
import re
HALL = re.compile("^(ご視聴ありがとうございました|チャンネル登録.+?|高評価.+?|次回もお楽しみに)+")
merged = []
for i, w in enumerate(wsegs):
    text = HALL.sub("", w["text"]).strip()
    if not text:
        continue
    spk, best = "SPEAKER_00", 0.0
    for d in dsegs:
        ov = max(0.0, min(w["end"], d["end"]) - max(w["start"], d["start"]))
        if ov > best:
            best, spk = ov, d["speaker"]
    merged.append({"index": len(merged), "start": round(w["start"], 2), "end": round(w["end"], 2),
                   "speaker": spk, "text": text,
                   "avg_logprob": w["avg_logprob"], "no_speech_prob": w["no_speech_prob"]})

import json
out = Path(OUTBASE); out.mkdir(parents=True, exist_ok=True)
stem = "align_" + Path(SRC).stem
(out / f"{stem}.json").write_text(json.dumps({"segments": merged}, ensure_ascii=False, indent=2), encoding="utf-8")

def ts(x):
    ms = max(0, int(x * 1000))
    return f"{ms//3600000:02d}:{(ms%3600000)//60000:02d}:{(ms%60000)//1000:02d},{ms%1000:03d}"

with open(out / f"{stem}.srt", "w", encoding="utf-8") as f:
    for m in merged:
        f.write(f"{m['index']+1}\n{ts(m['start'])} --> {ts(m['end'])}\n[{m['speaker']}] {m['text']}\n\n")
with open(out / f"{stem}.txt", "w", encoding="utf-8") as f:
    for m in merged:
        f.write(f"[{ts(m['start'])} {m['speaker']}] {m['text']}\n")
print("出力:", stem, f"{len(merged)}件")
print("出力先:", str(out))
if not USE_DRIVE:
    files.download(str(out / f"{stem}.json"))
    files.download(str(out / f"{stem}.txt"))


In [ ]:
# 合図: ブラウザで音を鳴らす (タブを開けて音量を上げること)
import numpy as np
from IPython.display import Audio, display
sr = 22050
t = np.linspace(0, 0.5, int(sr * 0.5), False)
tone = (0.5 * np.sin(2 * np.pi * 880 * t)).astype(np.float32)
for _ in range(3):
    display(Audio(tone, rate=sr, autoplay=True))
print("完了", flush=True)


## 仕上げ (Kai側)

1. 出たalign_txtをKaiに貼る。Pixel起こしtxtと突合せて誤変換と話者反転を直す。
2. 中国語部分の日本語化もKaiに頼む (whisperのtranslateは英語行きのみ、turboは不可のため)。
3. 話者数が崩れたらNUM_SPEAKERSを固定して分離セルだけ再実行。
